# RL4CRN app 17: Habituation Hallmarks Custom

This notebook runs the custom six-hallmark habituation objective implemented in `apps/habituation/hallmarks.py` through the RL4CRN training loop.

The task kind is registered as `habituation_hallmarks_custom`.


In [ ]:
import os, sys
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', '/tmp')

repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    if (candidate / 'RL4CRN').exists() and (candidate / 'apps').exists():
        repo_root = candidate
        break
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 1) Imports

In [ ]:
from RL4CRN.utils.input_interface import Configurator, make_task, make_session_and_trainer, print_task_summary
from RL4CRN.utils.default_tasks.HabituationHallmarksTaskKind import HabituationHallmarksCustomTaskKind
from apps.habituation.hallmarks_helpers import render_habituation

HabituationHallmarksCustomTaskKind.pretty_help()


## 2) Template IO/CRN

In [ ]:
from RL4CRN.utils.crn_builders import build_simple_IOCRN

cfg = Configurator.preset('paper')
cfg.solver.algorithm = 'CVODE'
cfg.solver.rtol = 1e-9
cfg.solver.atol = 1e-9

species_labels = ['X_1', 'X_2', 'X_3', 'X_4']
crn, species_labels = build_simple_IOCRN(
    species=species_labels,
    production_input_map={'X_1': 'u_1'},
    degradation_input_map={},
    dilution_map={},
    production_map={},
    output_species='X_4',
    solver=cfg.solver,
)

print('Template CRN built.')
print(crn)


## 3) Reaction Library

In [ ]:
from RL4CRN.utils.library_builders import build_MAK_library
from RL4CRN.iocrns.reaction_library import construct_catalytic_michaelis_mentin_library

library_components = build_MAK_library(crn, species_labels, order=1)
library, M, K, masks = library_components
cmm_library = construct_catalytic_michaelis_mentin_library(species_labels)
library.merge(cmm_library)

template_reaction_ids = {r.ID for r in crn.reactions}

bad_ids = [
    r.ID
    for r in library.reactions
    if r.reactant_labels == []
    and r.product_labels != []
    and r.ID not in template_reaction_ids
]
library.remove_reactions(bad_ids, remove_by='ID')

M = len(library.reactions)
K = library.get_num_parameters()
masks = {
    'continuous': library.get_parameter_mask(mode='continuous', force=True),
    'discrete': library.get_parameter_mask(mode='discrete', force=True),
    'logit': library.get_logit_mask(force=True),
}
crn.set_library_context(library)
library_components = library, M, K, masks

print('Added catalytic Michaelis-Menten reactions:', len(cmm_library.reactions))
print('Removed zero-order production reactions:', len(bad_ids))
print('Library built: M=', M, 'K=', K)


## 4) Custom Hallmark Task Parameters

In [ ]:
# Reference pulse protocol.
A = 10.0
T = 15.0
Ton = 1.11
n_pulses = 50

# Hallmark 4 and 5 sweeps.
T_values = [15.0, 20.0, 25.0]
A_values = [10.0, 20.0, 30.0]# Candidate input values for the RL task. The task uses A explicitly below,
# so this mainly keeps the RL4CRN task interface well-defined.
u_values = [A]

# Component weights.
hallmark_weights = {
    'hallmark1': 3.0,
    'hallmark2': 1.0,
    'hallmark3': 1.0,
    'hallmark4': 2.0,
    'hallmark5': 3.0,
    'hallmark6': 1.0,
}

# Per-loss keyword arguments.
h1_kwargs = {
    'tolerance': 0.01,
    'n_min': 10,
    'eps': 1e-12,
    'LARGE_NUMBER': 1e4,
}
h2_kwargs = {
    'recovery_tolerance': 0.05,
    'max_gap': 10000.0,
    'search_depth': 16,
}
h3_kwargs = {
    'n_series': 2,
    'recovery_gap_fraction': 0.5,
    'tolerance': 0.01,
}
h4_kwargs = {'tolerance': 0.01}
h5_kwargs = {'tolerance': 0.01}
h6_kwargs = {
    'stricter_recovery_tolerance': 0.01,
    'max_gap': 10000.0,
    'search_depth': 16,
}


## 5) Build Task

In [ ]:
task = make_task(
    template_crn=crn,
    library_components=library_components,
    kind='habituation_hallmarks_custom',
    species_labels=species_labels,
    params={
        'A': A,
        'T': T,
        'Ton': Ton,
        'n_pulses': n_pulses,
        'T_values': T_values,
        'A_values': A_values,
        'u_values': u_values,
        'ic': 'zero',
        'weights': hallmark_weights,
        'h1_kwargs': h1_kwargs,
        'h2_kwargs': h2_kwargs,
        'h3_kwargs': h3_kwargs,
        'h4_kwargs': h4_kwargs,
        'h5_kwargs': h5_kwargs,
        'h6_kwargs': h6_kwargs,
        'amplification_factors': {
            'hallmark1': 1.0,
            'hallmark2': 1.0,
            'hallmark3': 1.0,
            'hallmark4': 5.0,
            'hallmark5': 5.0,
            'hallmark6': 1.0,
        },
    },
)

print_task_summary(task)
assert len(task.u_list[0]) == crn.num_inputs


## 6) Training Configuration

In [ ]:
cfg.train.max_added_reactions = 8
cfg.train.epochs = 301
cfg.train.render_every = 5
cfg.train.seed = 0
cfg.train.hall_of_fame_size = 30
cfg.train.batch_size = 1280

cfg.agent.risk_scheduler = {'risk': 0.9, 'risk_update': 0.0, 'max_risk': 1.0, 'risk_schedule': 1000}
cfg.policy.entropy_weights_per_head = {"structure": 3.0, "continuous": 1.0, "discrete": 0.0, "input_influence": 0.0}
cfg.policy.continuous_distribution = {
    "type": "lognormal_independent",
}

cfg.render.n_best = 5
cfg.render.disregarded_percentage = 0.9
cfg.render.mode = {
    'style': 'logger',
    'task': 'habituation_hallmarks_custom',
    'format': 'figure',
    'figure_prefix': 'Custom Habituation',
    'figsize': (12, 30),
    'topology': True,
}


## 7) Optional Comet Logger and Trainer

In [ ]:
from datetime import datetime

logger = None
if os.environ.get('COMET_API_KEY') and os.environ.get('COMET_WORKSPACE'):
    from pytorch_lightning.loggers import CometLogger
    project_name = os.environ.get('COMET_PROJECT_NAME', 'Habituation_hallmarks_custom')
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    logger = CometLogger(
        api_key=os.environ['COMET_API_KEY'],
        project=project_name,
        workspace=os.environ['COMET_WORKSPACE'],
        name=f'{project_name}_{timestamp}',
    ).experiment
    logger.log_parameters({
        'A': A,
        'T': T,
        'Ton': Ton,
        'n_pulses': n_pulses,
        'T_values': T_values,
        'A_values': A_values,
        **{f'weight_{k}': v for k, v in hallmark_weights.items()},
    })

trainer = make_session_and_trainer(cfg, task, logger=logger)


## 8) Train

In [ ]:
checkpoint_path = 'habituation_hallmarks_custom_chkpt_run_Hab_CMM_4S_8R-5.pkl'
trainer.run(epochs=cfg.train.epochs, checkpoint_path=checkpoint_path)


## 9) Inspect and Render Best CRN

In [ ]:
best = trainer.inspect_best(plot=False)
if best is not None:
    print('Best loss:', best.last_task_info.get('reward'))
    print('Component losses:', best.last_task_info.get('component_losses'))
    fig, axes = render_habituation(best)
    fig.suptitle(f"Best CRN Custom Habituation Hallmarks, loss={best.last_task_info.get('reward'):.4g}")
    plt.show()
    if logger is not None:
        logger.log_figure(figure_name='Best CRN Custom Habituation Hallmarks', figure=fig)


## 10) Hall of Fame Rendering

In [ ]:
from RL4CRN.utils.visualizations import topology_graph

hof_envs = list(trainer.s.mult_env.hall_of_fame or [])[:10]
print('HoF entries plotted:', len(hof_envs))

for i, env in enumerate(hof_envs):
    crn_hof = env.state
    print(f'HoF {i} loss:', crn_hof.last_task_info.get('reward'))
    fig, _ = render_habituation(crn_hof)
    fig.suptitle(f'HoF {i} Custom Habituation Hallmarks')
    plt.show()
    if logger is not None:
        logger.log_figure(figure_name=f'Custom Habituation HoF {i}', figure=fig)

if hof_envs:
    fig_hof_div = topology_graph([env.state for env in hof_envs], t=5, figsize=(10, 10))
    fig_hof_div.suptitle('Custom Habituation HoF Top-10 Diversity Graph')
    plt.show()
    if logger is not None:
        logger.log_figure(figure_name='Custom Habituation HoF topology diversity', figure=fig_hof_div)


## 11) Save Final Checkpoint

In [ ]:
trainer.save(checkpoint_path)

- Currently running:
    - run_3S-2: increase w_4 from 2 to 5, max number of reactions increased from 5 to 8.

- To test:
    2- run_3S_dilution-1: added dilution reactions. Same conditions as 1.
